# CDC-CP P1 sweep: every backbone, then every head, on two GPUs

One notebook finds the best P1 backbone and projection head. Every run uses the best P1 recipe: `configs/p1_lr1e-4.yaml`, which is backbone LR 1e-4, heads 1e-3, 20 epochs, batch 32 and Baseline 1 augmentation. Only the backbone or the head changes, so settings stay consistent across runs, as Dr. Lee asked.

| Phase | What runs | Runs |
|---|---|---|
| **A** | Each backbone with the standard head (1280 → 256 → 128, ReLU), seed 42: EfficientNet-B0/B2/B3, ResNet50, MobileNetV3-Large, ConvNeXt-Tiny | 6 |
| **B** | Head variants on the 2 best backbones from A: linear (→128), 512, 512 → 256, 256 with GELU | 8 |
| **C** | Seed 43 of the 3 best configs from A and B, so the winner is not a lucky seed | 3 |

**Ranking (`scripts/calib_board.py`).** Each run is scored on its last 3 epochs:
- **Raw:** no seen-pair bias. This is the plain row every model reports.
- **Keep-seen (the ranking column):** the best unseen accuracy whose seen accuracy stays within 3 points of raw. This follows Dr. Lee's rule that unseen should not improve at a large cost to seen.
- **Best-H and AUC:** shown for the record.

It also writes the γ and r sensitivity table (the calibration ablation).

**Setup (once):** inputs are the code dataset and the data dataset, as in round 3. Then **Accelerator → GPU T4 x2** and **Internet → On**.

**Run unattended:** **Save Version → Save & Run All (Commit)**. It takes about 3-4 hours, and two runs train at once, one per GPU.

**If it stops early:** Add Input → this notebook's last version (its output), then Save & Run All again. Finished runs are skipped and an interrupted run resumes.

Only `DPD_Small_train.csv` and its 8,000 images are read. The official test files are never opened.

## 1. Settings

In [ ]:
from pathlib import Path

SMOKE    = False   # True: 1 epoch of every phase-A backbone (about 10 min), to check everything builds on the GPU
TOL      = 3.0     # keep-seen: seen accuracy may fall at most this many points below raw
TOP_B    = 2       # backbones carried into phase B
TOP_C    = 3       # configs re-run with seed 43 in phase C
WORKERS  = 2       # data-loader workers per run (2 runs share Kaggle's 4 CPU cores)

BACKBONES = ['efficientnet_b0', 'efficientnet_b2', 'efficientnet_b3', 'resnet50', 'mobilenet_v3_large', 'convnext_tiny']
HEADS = {                       # tag: (hidden widths, activation)
    '256':     ([256], 'relu'),  # the team plan's head, used in phase A
    'lin':     ([], 'relu'),
    '512':     ([512], 'relu'),
    '512-256': ([512, 256], 'relu'),
    '256gelu': ([256], 'gelu'),
}
BASE_CONFIG = 'configs/p1_lr1e-4.yaml'

INPUT     = Path('/kaggle/input')
RUNS_DIR  = Path('/kaggle/working/sweep_smoke' if SMOKE else '/kaggle/working/runs')
CODE_DIR  = Path('/tmp/cdc-cp')
DATA_DIR  = Path('/tmp/DPD_Small')
CACHE_DIR = Path('/tmp/cache')

## 2. GPUs and code

In [ ]:
import os, shutil, subprocess, sys, zipfile, time
import torch

assert torch.cuda.is_available(), 'Settings > Accelerator > GPU T4 x2'
N_GPU = torch.cuda.device_count()
print(N_GPU, 'x', torch.cuda.get_device_name(0), '| torch', torch.__version__)
if N_GPU < 2:
    print('Only one GPU: runs will go one at a time (about twice as long). Pick GPU T4 x2 to run two at once.')

if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
found = next((p.parent.parent for p in INPUT.rglob('cdccp/__init__.py')), None)
if found is not None:
    shutil.copytree(found, CODE_DIR)
else:
    code_zip = next(INPUT.rglob('cdc-cp_code*.zip'), None)
    assert code_zip, 'Add the code dataset (cdc-cp_code.zip) as an input'
    with zipfile.ZipFile(code_zip) as z:
        z.extractall('/tmp')
assert (CODE_DIR / 'scripts' / 'calib_board.py').is_file(), 'old code dataset: upload the new cdc-cp_code.zip (it must contain scripts/calib_board.py)'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'], check=True)
os.chdir(CODE_DIR)
sys.path.insert(0, str(CODE_DIR))
from cdccp.models import BACKBONES as KNOWN
assert set(BACKBONES) <= set(KNOWN), f'code does not know {set(BACKBONES) - set(KNOWN)}: upload the new code zip'
print('code from', found or code_zip)


def sh(*args):
    proc = subprocess.Popen([str(a) for a in args], cwd=CODE_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    if proc.wait():
        raise RuntimeError(f'exit code {proc.returncode}: {args}')

## 3. Training data (train CSV and its 8,000 images only)

In [ ]:
import hashlib
import pandas as pd

train_csv = next(p for p in INPUT.rglob('*') if p.name.lower() == 'dpd_small_train.csv')
cols = ['filename', 'crop_id', 'disease_id', 'composition_id', 'crop_name', 'disease_name', 'composition_name']
train = pd.read_csv(train_csv, header=None, names=cols)
assert len(train) == 8000, f'expected 8,000 rows, got {len(train)}'
digest = hashlib.sha256(train_csv.read_bytes()).hexdigest()
assert digest.startswith('9f1b729a'), f'train CSV hash {digest[:8]} does not match the split manifest'

first = train.filename.iloc[0]
image_src = next((d for d in [train_csv.parent, *train_csv.parent.parent.rglob('*')] if d.is_dir() and (d / first).is_file()), None)
assert image_src is not None, f'could not find {first} near {train_csv.parent}'
DATA_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(train_csv, DATA_DIR / 'DPD_Small_train.csv')
link = DATA_DIR / 'DPD_Small_dataset'
if not link.exists():
    link.symlink_to(image_src, target_is_directory=True)
missing = [n for n in train.filename if not (link / n).is_file()]
assert not missing, f'{len(missing)} images missing, e.g. {missing[:3]}'
os.environ['DPD_ROOT'] = str(DATA_DIR)
print(f'8,000 training images linked from {image_src}')

## 4. Restore runs from an earlier version's output

In [ ]:
RUNS_DIR.mkdir(parents=True, exist_ok=True)
restored = []
for last in INPUT.rglob('sweep_*_seed*/last.pt'):
    src, dst = last.parent, RUNS_DIR / last.parent.name
    old = dst / 'last.pt'
    if not old.exists() or torch.load(old, map_location='cpu', weights_only=False)['applied'] < torch.load(last, map_location='cpu', weights_only=False)['applied']:
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(src, dst)
        restored.append(dst.name)
print('restored:', restored or 'nothing (first version)')

## 5. Image cache (local disk, a few minutes)

In [ ]:
sh(sys.executable, 'scripts/build_cache.py', '--verify', '--workers', 8, f'paths.cache_dir={CACHE_DIR}')

## 6. Run queue
Each GPU takes the next waiting run. A run's log lines are printed as it trains, prefixed with its name. When a run finishes, the scoreboard can read it. Only its last 3 epoch checkpoints are kept, which stays inside Kaggle's 20 GB output limit.

In [ ]:
COMMON = [f'paths.runs_dir={RUNS_DIR}', f'paths.cache_dir={CACHE_DIR}', f'loader.num_workers={WORKERS}']


def run_name(backbone, head, seed):
    return f'sweep_{backbone}_h{head}_seed{seed}'


def job(backbone, head, seed=42):
    hidden, act = HEADS[head]
    name = run_name(backbone, head, seed)[:-len(f'_seed{seed}')]
    args = [sys.executable, 'scripts/train.py', BASE_CONFIG, *COMMON, f'name={name}', f'model.backbone={backbone}',
            f'model.hidden={hidden}', f'model.activation={act}', f'seed={seed}', '--resume']
    if SMOKE:
        args += ['--max-epochs', '1']
    return run_name(backbone, head, seed), args


def prune(run_dir, keep=3):
    pts = sorted(run_dir.glob('epoch_*.pt'))
    for p in pts[:-keep]:
        p.unlink()


def run_queue(jobs):
    """Run (name, args) jobs, one per GPU at a time. A failed run is reported and left out of the ranking."""
    waiting, running, failed = list(jobs), {}, []
    print('queue:', [n for n, _ in waiting], flush=True)
    while waiting or running:
        for gpu in range(max(N_GPU, 1)):
            if gpu not in running and waiting:
                name, args = waiting.pop(0)
                log = open(RUNS_DIR / f'{name}.log', 'a')
                env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
                p = subprocess.Popen([str(a) for a in args], cwd=CODE_DIR, env=env, stdout=log, stderr=subprocess.STDOUT, text=True)
                running[gpu] = (name, p, log, 0, time.time())
                print(f'[gpu {gpu}] start {name}', flush=True)
        time.sleep(20)
        for gpu, (name, p, log, seen, t0) in list(running.items()):
            log.flush()
            lines = (RUNS_DIR / f'{name}.log').read_text(errors='replace').splitlines()
            for line in lines[seen:]:
                if line.startswith(('epoch', 'done', 'Traceback')) or 'Error' in line:
                    print(f'[gpu {gpu}] {name}: {line}', flush=True)
            running[gpu] = (name, p, log, len(lines), t0)
            if p.poll() is not None:
                log.close()
                del running[gpu]
                mins = (time.time() - t0) / 60
                if p.returncode:
                    failed.append(name)
                    print(f'[gpu {gpu}] FAILED {name} after {mins:.0f} min; last lines:', *lines[-15:], sep='\n', flush=True)
                else:
                    print(f'[gpu {gpu}] finished {name} in {mins:.0f} min', flush=True)
                    if (RUNS_DIR / name).is_dir():
                        prune(RUNS_DIR / name)
    if failed:
        print(f'WARNING failed runs, left out of the ranking: {failed} (logs in {RUNS_DIR})', flush=True)
    return failed


def board(*patterns):
    sh(sys.executable, 'scripts/calib_board.py', RUNS_DIR, '--tol', TOL, '--match', *patterns)
    return pd.read_csv(RUNS_DIR / 'calib_board.csv')


def ranked_configs(df):
    """Configs (backbone, head tag) by mean keep-seen H, best first."""
    df = df.assign(head=df.config.str.extract(r'_h([^_]+)$')[0])
    return df.groupby(['backbone', 'head'])['H keep'].mean().sort_values(ascending=False)

## 7. Phase A: every backbone with the standard head

In [ ]:
run_queue([job(b, '256') for b in BACKBONES])
if SMOKE:
    print('SMOKE run finished: every backbone trains on this GPU. Set SMOKE = False and Save & Run All (Commit).')
else:
    a = ranked_configs(board('sweep_*_h256_seed42'))
    print(a)
    TOP_BACKBONES = list(a.index.get_level_values('backbone')[:TOP_B])
    print('phase B backbones:', TOP_BACKBONES)

## 8. Phase B: head variants on the best backbones

In [ ]:
if not SMOKE:
    run_queue([job(b, h) for b in TOP_BACKBONES for h in HEADS if h != '256'])
    ab = ranked_configs(board('sweep_*_seed42'))
    print(ab)
    TOP_CONFIGS = list(ab.index[:TOP_C])
    print('phase C configs:', TOP_CONFIGS)

## 9. Phase C: second seed for the finalists

In [ ]:
if not SMOKE:
    run_queue([job(b, h, seed=43) for b, h in TOP_CONFIGS])

## 10. Final scoreboard
Saved as `runs/calib_board.md`, `calib_board.csv` and `gamma_sensitivity.csv` in this version's output. Configs with 2 seeds show a +/- spread; a gap smaller than the spread is not a real difference.

In [ ]:
if not SMOKE:
    final = board('sweep_*')
    from IPython.display import Markdown, display
    display(Markdown((RUNS_DIR / 'calib_board.md').read_text(encoding='utf-8')))

## 11. Curves of the finalists

In [ ]:
if not SMOKE:
    import matplotlib.pyplot as plt
    names = [run_name(b, h, s) for b, h in TOP_CONFIGS for s in (42, 43)]
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    for n in names:
        h = pd.read_csv(RUNS_DIR / n / 'history.csv')
        line, = axes[0].plot(h.epoch, h.A_S, label=n)
        axes[1].plot(h.epoch, h.unseen_disease_top1, color=line.get_color(), label=n)
        axes[2].plot(h.epoch, h.val_ce_disease, color=line.get_color(), label=n)
    for ax, t in zip(axes, ['Seen pairs Top-1 (raw)', 'Unseen disease Top-1', 'Validation disease cross-entropy (seen)']):
        ax.set_title(t); ax.set_xlabel('epoch'); ax.grid(alpha=.3); ax.legend(fontsize=6)
    plt.tight_layout(); plt.show()